In [1]:
import os
from tqdm import tqdm
import csv
from libs.character_search import client, calculate_gender_frequencies

In [2]:
game_name = "arknights"
max_pages = 1000
hide_empty = True

In [3]:
output_dir = "outputs"
os.makedirs(output_dir, exist_ok=True)

page = 1
limit = 1000  # 每页最大记录数
all_character_tags = []

print(f"正在获取{game_name}的角色数据...")
with tqdm(total=max_pages, desc="获取角色列表") as pbar:
    while page <= max_pages:
        character_tags = client.tag_list(
            name_matches=f"*{game_name}*",
            category=4,
            order="count",
            page=page,
            limit=limit,
            hide_empty=hide_empty,
        )

        # # 打印第一个tag的内容以检查结构
        # if character_tags and page == 1:
        #     print("\nDebug - First tag content:")
        #     print(character_tags[0])

        if not character_tags:
            break

        all_character_tags.extend(character_tags)
        page += 1
        pbar.update(1)

正在获取arknights的角色数据...


获取角色列表:   0%|          | 2/1000 [00:02<20:14,  1.22s/it]


In [4]:
# 第一阶段：分析角色性别并保存在内存中
filtered_characters = [
    tag
    for tag in all_character_tags
    if game_name.lower() in tag.get("name", "").lower()
]

print("\n正在分析角色性别...")
analyzed_characters = []

for tag in tqdm(filtered_characters, desc="分析角色性别"):
    name = tag.get("name", "")
    post_count = tag.get("post_count", 0)
    created_at = tag.get("created_at", "")
    updated_at = tag.get("updated_at", "")

    try:
        # 使用新的函数计算性别频率
        (
            male_releted_tags_frequency_avg,
            female_releted_tags_frequency_avg,
        ) = calculate_gender_frequencies(name)

        # 保存角色数据到内存
        character_data = {
            "name": name,
            "male_frequency_avg": male_releted_tags_frequency_avg,
            "female_frequency_avg": female_releted_tags_frequency_avg,
            "post_count": post_count,
            "created_at": created_at,
            "updated_at": updated_at,
        }
        analyzed_characters.append(character_data)

    except Exception as e:
        print(f"\n获取角色 {name} 的相关tag时出错: {e}")
        # 错误情况也保存到内存
        character_data = {
            "name": name,
            "male_frequency_avg": 0,
            "female_frequency_avg": 0,
            "post_count": post_count,
            "created_at": created_at,
            "updated_at": updated_at,
            "gender": "unknown",
        }
        analyzed_characters.append(character_data)


正在分析角色性别...


分析角色性别: 100%|██████████| 1353/1353 [08:51<00:00,  2.55it/s]


In [5]:
print(analyzed_characters)

[{'name': 'doctor_(arknights)', 'male_frequency_avg': 0.3174, 'female_frequency_avg': 0.3506666666666667, 'post_count': 9602, 'created_at': '2019-06-11T22:25:59.418-04:00', 'updated_at': '2019-09-01T22:13:14.906-04:00'}, {'name': 'amiya_(arknights)', 'male_frequency_avg': 0.1102, 'female_frequency_avg': 0.3824, 'post_count': 8623, 'created_at': '2017-10-16T12:54:22.106-04:00', 'updated_at': '2019-09-02T04:48:19.806-04:00'}, {'name': 'texas_(arknights)', 'male_frequency_avg': 0.1348, 'female_frequency_avg': 0.44915, 'post_count': 7075, 'created_at': '2017-10-16T14:27:27.292-04:00', 'updated_at': '2019-09-02T05:38:00.259-04:00'}, {'name': 'skadi_(arknights)', 'male_frequency_avg': 0.155, 'female_frequency_avg': 0.3708, 'post_count': 5871, 'created_at': '2019-06-11T12:51:45.708-04:00', 'updated_at': '2019-09-02T01:52:11.512-04:00'}, {'name': 'lappland_(arknights)', 'male_frequency_avg': 0.1348, 'female_frequency_avg': 0.45010000000000006, 'post_count': 5454, 'created_at': '2019-05-22T15:1

In [6]:
# 第二阶段：对角色进行去重
analyzed_characters_copy = []
for character in analyzed_characters:
    # 复制角色数据到新的列表
    analyzed_characters_copy.append(character.copy())

print("\n正在去重角色数据...")
all_character_name = []
for character in analyzed_characters:
    all_character_name.append(character["name"])

character_dict = {}
# print(analyzed_characters[0])
while len(analyzed_characters_copy) >= 1:
    character_data = []
    character_data.append(analyzed_characters_copy.pop(0))  # 获取第一个角色数据
    character = character_data[0]["name"]  # 获取角色名称
    character_name = character.split("_(")[0]  # 去除括号及其内容
    # print(f"正在处理角色: {character}, 剩余角色数: {len(analyzed_characters_copy)}")

    for a_character in analyzed_characters_copy:
        if a_character["name"].split("_(")[0] == character_name:
            # print(f"找到重复角色: {a_character['name']}")
            # 从列表中移除重复的角色
            analyzed_characters_copy.remove(a_character)
            character_data.append(a_character)
    # print(character_data)
    # 根据count加权计算频率
    female_frequency = 0
    male_frequency = 0
    for data in character_data:
        female_frequency += data["female_frequency_avg"] * data["post_count"]
        male_frequency += data["male_frequency_avg"] * data["post_count"]
    total_count = sum(data["post_count"] for data in character_data)
    female_frequency = female_frequency / total_count
    male_frequency = male_frequency / total_count
    # 根据频率判断性别
    if abs(female_frequency - male_frequency) < 0.05:
        gender = "unknown"
    elif female_frequency > male_frequency:
        gender = "female"
    else:
        gender = "male"

    # 保存去重后的角色数据
    for data in character_data:
        character_dict[data["name"]] = {
            "name": data["name"],
            "male_frequency_avg": male_frequency,
            "female_frequency_avg": female_frequency,
            "post_count": data["post_count"],
            "created_at": data["created_at"],
            "updated_at": data["updated_at"],
            "gender": gender,
        }


正在去重角色数据...


In [7]:
print(all_character_name[0])
print(character_dict)
print(character_dict[all_character_name[0]])

doctor_(arknights)
{'doctor_(arknights)': {'name': 'doctor_(arknights)', 'male_frequency_avg': 0.3174, 'female_frequency_avg': 0.3506666666666667, 'post_count': 9602, 'created_at': '2019-06-11T22:25:59.418-04:00', 'updated_at': '2019-09-01T22:13:14.906-04:00', 'gender': 'unknown'}, 'amiya_(arknights)': {'name': 'amiya_(arknights)', 'male_frequency_avg': 0.11196200502221365, 'female_frequency_avg': 0.39607416135470996, 'post_count': 8623, 'created_at': '2017-10-16T12:54:22.106-04:00', 'updated_at': '2019-09-02T04:48:19.806-04:00', 'gender': 'female'}, 'amiya_(medic)_(arknights)': {'name': 'amiya_(medic)_(arknights)', 'male_frequency_avg': 0.11196200502221365, 'female_frequency_avg': 0.39607416135470996, 'post_count': 390, 'created_at': '2024-04-30T04:52:56.399-04:00', 'updated_at': '2024-04-30T04:52:56.399-04:00', 'gender': 'female'}, 'amiya_(guard)_(arknights)': {'name': 'amiya_(guard)_(arknights)', 'male_frequency_avg': 0.11196200502221365, 'female_frequency_avg': 0.39607416135470996,

In [8]:
def save_characters_to_files(
    analyzed_characters, character_dict, game_name, output_dir
):
    """
    将分析好的角色数据保存到CSV文件
    :param analyzed_characters: 已分析的角色数据列表
    :param game_name: 游戏名称
    :param output_dir: 输出目录
    """
    # 创建文件名
    male_filename = os.path.join(output_dir, f"{game_name}_male_characters.csv")
    female_filename = os.path.join(output_dir, f"{game_name}_female_characters.csv")
    unknown_filename = os.path.join(
        output_dir, f"{game_name}_unknown_gender_characters.csv"
    )

    # 按性别分组数据
    male_characters = []
    for c in analyzed_characters:
        if character_dict[c]["gender"] == "male":
            male_characters.append(character_dict[c])
    female_characters = []
    for c in analyzed_characters:
        if character_dict[c]["gender"] == "female":
            female_characters.append(character_dict[c])
    unknown_characters = []
    for c in analyzed_characters:
        if character_dict[c]["gender"] == "unknown":
            unknown_characters.append(character_dict[c])

    # CSV头部
    headers = [
        "name",
        "male_frequency_avg",
        "female_frequency_avg",
        "post_count",
        "created_at",
        "updated_at",
    ]

    # 写入男性角色文件
    with open(male_filename, "w", encoding="utf-8", newline="") as male_file:
        male_writer = csv.writer(male_file)
        male_writer.writerow(headers)
        for character in male_characters:
            row_data = [
                character["name"],
                character["male_frequency_avg"],
                character["female_frequency_avg"],
                character["post_count"],
                character["created_at"],
                character["updated_at"],
            ]
            male_writer.writerow(row_data)

    # 写入女性角色文件
    with open(female_filename, "w", encoding="utf-8", newline="") as female_file:
        female_writer = csv.writer(female_file)
        female_writer.writerow(headers)
        for character in female_characters:
            row_data = [
                character["name"],
                character["male_frequency_avg"],
                character["female_frequency_avg"],
                character["post_count"],
                character["created_at"],
                character["updated_at"],
            ]
            female_writer.writerow(row_data)

    # 写入未知性别角色文件
    with open(unknown_filename, "w", encoding="utf-8", newline="") as unknown_file:
        unknown_writer = csv.writer(unknown_file)
        unknown_writer.writerow(headers)
        for character in unknown_characters:
            row_data = [
                character["name"],
                character["male_frequency_avg"],
                character["female_frequency_avg"],
                character["post_count"],
                character["created_at"],
                character["updated_at"],
            ]
            unknown_writer.writerow(row_data)


# 第三阶段：将内存中的数据写入文件
print("\n正在保存角色数据到文件...")
save_characters_to_files(all_character_name, character_dict, game_name, output_dir)


正在保存角色数据到文件...
